# インボイス確認ツール(Colab 版)

[GitHub Pages 版](https://masakiniwa.github.io/Invoice/)と同じ画面を、Google Colab の計算資源(GPU)で動かします。

- 文字認識(PaddleOCR)を Colab の **GPU** で実行します。ブラウザにモデルをダウンロードしないので、スマホでも軽く動きます
- 起動すると、アプリを開くリンクと、毎回変わる **パスワード** が表示されます
- 読み込んだ画像は Colab のメモリ上で処理し、保存しません(履歴・設定はこれまでどおり使う人のブラウザに保存されます)

## 使い方
1. メニューの「ランタイム」→「ランタイムのタイプを変更」で **T4 GPU** を選ぶ(CPU でも動きます)
2. 下のセルを上から順に実行する(▶ ボタン)
3. 表示された「アプリを開く」からログインする

## リンクは 2 種類
| | Colab 内のリンク(既定) | 公開リンク(Cloudflare) |
| --- | --- | --- |
| 開ける人 | このノートブックを開いている **Google アカウントでログインしたブラウザだけ** | リンクとパスワードを知っている人なら誰でも |
| 向いている使い方 | 自分で使う(より安全) | 同僚の端末など、別のアカウント・端末から使う |

> どちらも、ノートブックを止めるかランタイムが切断されると使えなくなります。パスワード入りのリンクは人に送らないでください。

In [ ]:
#@title 設定 { display-mode: "form" }
#@markdown **UI の取得方法**: `pages` = 公開中の版 / `pages-dev` = 開発版(dev ブランチ)/ `build` = 下のブランチのソースからビルド(push 前の確認など。数分かかります)
SOURCE = "pages"  #@param ["pages", "pages-dev", "build"]
#@markdown リポジトリのブランチ(サーバーのプログラムもこのブランチのものを使います。開発版を試すときは `dev`)
BRANCH = "main"  #@param {type:"string"}
#@markdown 公開リンク(Cloudflare)も作る(別のアカウント・端末から使うとき。自分だけなら不要)
USE_TUNNEL = False  #@param {type:"boolean"}
REPO = "https://github.com/MasakiNiwa/Invoice.git"

In [ ]:
#@title 1. 準備(初回 1〜2 分) { display-mode: "form" }
import os, subprocess, sys
REPO_DIR = "/content/Invoice"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO, REPO_DIR], check=True)
sys.path.insert(0, os.path.join(REPO_DIR, "colab"))
import importlib, launcher
importlib.reload(launcher)
print("GPU:", "あり" if launcher.has_gpu() else "なし(CPU で動かします)")
launcher.setup(source=SOURCE)

In [ ]:
#@title 2. 起動(リンクとパスワードを表示) { display-mode: "form" }
#@markdown つながることを確かめてからリンクを表示します。起動後も裏で見張り、止まったら自動で起動し直します。
info = launcher.start(tunnel=USE_TUNNEL)

### 困ったとき
- 開けない・エラーになる: 下の「診断」を実行して、結果(パスワードは表示されません)を共有してください
- 「Error 1033」: 公開リンクの接続が切れています。「2. 起動」をもう一度実行すると、新しいリンクで起動し直します
- スマホで使うとき: Colab のタブを閉じたり長く放置するとランタイムが切断されることがあります

In [ ]:
#@title 診断 { display-mode: "form" }
launcher.diagnose()

In [ ]:
#@title 停止 { display-mode: "form" }
launcher.stop()